<a href="https://colab.research.google.com/github/HaoLe2710/weather-streaming-bigdata/blob/model%2Fweather-forecast-xgboost-v1/notebooks/weather_forecast_xgboost_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weather Forecast Modeling V1 — Global XGBoost vs Persistence

This Colab run trains one global model for same-location temperature at `t + 1 hour`. It selects candidates on VALIDATION 2024 only, freezes the final TRAIN+VALIDATION model and hash, then evaluates untouched TEST 2025 once. The official persistence prediction is `temperature_c(t)`.

The code and feature contract are frozen by the repository commit recorded below. Do not use TEST metrics to tune or change the candidate.


## 1. Repository and frozen code commit

In [1]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime, timezone

REPOSITORY_URL = "https://github.com/HaoLe2710/weather-streaming-bigdata.git"
EXPECTED_BRANCH = "model/weather-forecast-xgboost-v1"
REPOSITORY_DIR = Path("/content/weather-streaming-bigdata")

if not REPOSITORY_DIR.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", EXPECTED_BRANCH], cwd=REPOSITORY_DIR, check=True)
subprocess.run(["git", "checkout", EXPECTED_BRANCH], cwd=REPOSITORY_DIR, check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", EXPECTED_BRANCH], cwd=REPOSITORY_DIR, check=True)

TRAINING_CODE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, text=True
).strip()
CURRENT_BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPOSITORY_DIR, text=True
).strip()
GIT_STATUS = subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=REPOSITORY_DIR, text=True
).strip()
assert CURRENT_BRANCH == EXPECTED_BRANCH, CURRENT_BRANCH
assert not GIT_STATUS, f"Colab checkout must be clean before the run: {GIT_STATUS}"
os.chdir(REPOSITORY_DIR)
sys.path.insert(0, str(REPOSITORY_DIR))
print({"branch": CURRENT_BRANCH, "training_code_commit": TRAINING_CODE_COMMIT, "git_status": "clean"})


{'branch': 'model/weather-forecast-xgboost-v1', 'training_code_commit': 'aea82dd33359087fc7adb115a778a0e3b86e7e9a', 'git_status': 'clean'}


## 2. Google Drive authorization and stable paths

In [2]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
DATASET_DIR = DRIVE_MYDRIVE / "weather-streaming-bigdata/datasets/weather_forecast_fe_v1"
RUNS_ROOT = DRIVE_MYDRIVE / "weather-streaming-bigdata/artifacts/weather_forecast_xgboost_v1"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f"Upload the frozen feature folder to {DATASET_DIR} before continuing. "
        "Do not regenerate or substitute the dataset."
    )
print({"dataset_dir": str(DATASET_DIR), "run_root": str(RUNS_ROOT)})


MessageError: Error: credential propagation was unsuccessful

## 3. Run ID and resume protection

In [ ]:
from ml.artifacts import atomic_write_json, atomic_write_text, read_json, sha256_file

FORCE_NEW_RUN = False  # Set True only when intentionally starting a fresh full experiment.
ACTIVE_RUN_FILE = RUNS_ROOT / "ACTIVE_RUN_ID.txt"

def new_run_id():
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    base = f"{timestamp}-xgb-v1"
    candidate = base
    suffix = 2
    while (RUNS_ROOT / candidate).exists():
        candidate = f"{base}-{suffix:02d}"
        suffix += 1
    return candidate

if ACTIVE_RUN_FILE.exists() and not FORCE_NEW_RUN:
    RUN_ID = ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip()
    RUN_DIR = RUNS_ROOT / RUN_ID
    if (RUN_DIR / "training_manifest.json").exists():
        saved_manifest = read_json(RUN_DIR / "training_manifest.json")
        if saved_manifest.get("status") == "COMPLETE":
            raise RuntimeError(
                f"Run {RUN_ID} is already complete and immutable. Set FORCE_NEW_RUN=True to start another run."
            )
    PREVIOUS_RUN_ID = None
    CODE_COMMIT_RECORD = RUN_DIR / "training_code_commit.json"
    if CODE_COMMIT_RECORD.exists():
        saved_commit = read_json(CODE_COMMIT_RECORD)
        if saved_commit.get("training_code_commit") != TRAINING_CODE_COMMIT:
            PREVIOUS_RUN_ID = RUN_ID
            RUN_ID = new_run_id()
            atomic_write_text(ACTIVE_RUN_FILE, RUN_ID + "\n")
            RUN_DIR = RUNS_ROOT / RUN_ID
            RUN_DIR.mkdir(parents=True, exist_ok=False)
else:
    PREVIOUS_RUN_ID = None
    RUN_ID = new_run_id()
    atomic_write_text(ACTIVE_RUN_FILE, RUN_ID + "\n")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)

RUN_DIR.mkdir(parents=True, exist_ok=True)
CODE_COMMIT_RECORD = RUN_DIR / "training_code_commit.json"
if CODE_COMMIT_RECORD.exists():
    saved_commit = read_json(CODE_COMMIT_RECORD)
    if saved_commit.get("training_code_commit") != TRAINING_CODE_COMMIT:
        raise RuntimeError("Code commit changed during resume; start a new run after freezing code.")
else:
    atomic_write_json(
        CODE_COMMIT_RECORD,
        {"training_code_commit": TRAINING_CODE_COMMIT, "branch": CURRENT_BRANCH, "git_status": "clean"},
    )
print({"run_id": RUN_ID, "run_dir": str(RUN_DIR), "previous_run_id": PREVIOUS_RUN_ID, "resuming": (RUN_DIR / "candidate_results.json").exists()})


## 4. Environment dependencies

In [ ]:
PACKAGE_PINS = []
SAVED_ENVIRONMENT_PATH = RUN_DIR / "training_environment.json"
if SAVED_ENVIRONMENT_PATH.exists():
    saved_environment = read_json(SAVED_ENVIRONMENT_PATH)
    for distribution, key in (
        ("xgboost", "xgboost_version"),
        ("numpy", "numpy_version"),
        ("pandas", "pandas_version"),
        ("pyarrow", "pyarrow_version"),
        ("scikit-learn", "sklearn_version"),
    ):
        PACKAGE_PINS.append(f"{distribution}=={saved_environment[key]}")
INSTALL_COMMAND = [sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-ml.txt", *PACKAGE_PINS]
subprocess.run(INSTALL_COMMAND, cwd=REPOSITORY_DIR, check=True)
print({"requirements_file": str(REPOSITORY_DIR / "requirements-ml.txt"), "resume_version_pins": PACKAGE_PINS})

import importlib.metadata
import json
import os
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import xgboost as xgb


## 5. Load and verify the frozen feature contract and dataset

In [ ]:
from ml.config import (
    EXPECTED_FINAL_TRAINING_ROWS,
    EXPECTED_LOCATION_COUNT,
    EXPECTED_ROWS_PER_TEST_LOCATION,
    EXPECTED_SPLIT_ROWS,
    FEATURE_ARTIFACT_RELATIVE_PATH,
    RANDOM_SEED,
    TARGET_COLUMN,
)
from ml.data_loader import (
    feature_list_artifact,
    load_feature_contract,
    load_split_arrays,
    verify_dataset,
)
from ml.artifacts import atomic_write_json

ARTIFACT_DIR = REPOSITORY_DIR / FEATURE_ARTIFACT_RELATIVE_PATH
FEATURE_CONTRACT = load_feature_contract(ARTIFACT_DIR)
atomic_write_json(RUN_DIR / "model_feature_list.json", feature_list_artifact(FEATURE_CONTRACT))
DATASET_VERIFICATION = verify_dataset(DATASET_DIR, FEATURE_CONTRACT)
atomic_write_json(RUN_DIR / "dataset_verification.json", DATASET_VERIFICATION)
assert len(FEATURE_CONTRACT.model_features) == 73
assert DATASET_VERIFICATION["rows_by_split"] == EXPECTED_SPLIT_ROWS
assert DATASET_VERIFICATION["test_values_materialized"] is False
print({
    "feature_set_id": FEATURE_CONTRACT.feature_set_id,
    "feature_count": len(FEATURE_CONTRACT.model_features),
    "feature_list_sha256": FEATURE_CONTRACT.feature_list_sha256,
    "rows_by_split": DATASET_VERIFICATION["rows_by_split"],
    "parquet_files": DATASET_VERIFICATION["parquet_file_count"],
    "parquet_bytes": DATASET_VERIFICATION["parquet_bytes"],
    "test_values_materialized": False,
})


## 6. Record Colab hardware and runtime libraries

In [ ]:
from ml.xgboost_model import detect_accelerator

def available_ram_bytes():
    try:
        import psutil
        return int(psutil.virtual_memory().total)
    except ImportError:
        with open("/proc/meminfo", "r", encoding="utf-8") as handle:
            for line in handle:
                if line.startswith("MemTotal:"):
                    return int(line.split()[1]) * 1024
    return None

ACCELERATOR = detect_accelerator()
CURRENT_RUNTIME_VERSIONS = {
    "python_version": platform.python_version(),
    "xgboost_version": xgb.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "pyarrow_version": pyarrow.__version__,
    "sklearn_version": importlib.metadata.version("scikit-learn"),
}
ENVIRONMENT_PATH = RUN_DIR / "training_environment.json"
if ENVIRONMENT_PATH.exists():
    TRAINING_ENVIRONMENT = read_json(ENVIRONMENT_PATH)
    for name, version in CURRENT_RUNTIME_VERSIONS.items():
        if TRAINING_ENVIRONMENT.get(name) != version:
            raise RuntimeError(
                f"Runtime version changed during resume ({name}: "
                f"{TRAINING_ENVIRONMENT.get(name)} -> {version}); do not mix candidate environments."
            )
    TRAINING_ENVIRONMENT.setdefault("resume_sessions", []).append({
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
        "device_probe": ACCELERATOR,
        "cpu_count": os.cpu_count(),
        "ram_bytes": available_ram_bytes(),
    })
else:
    TRAINING_ENVIRONMENT = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "training_code_commit": TRAINING_CODE_COMMIT,
        **CURRENT_RUNTIME_VERSIONS,
        "platform": platform.platform(),
        "device_probe": ACCELERATOR,
        "cpu_count": os.cpu_count(),
        "ram_bytes": available_ram_bytes(),
        "resume_sessions": [],
    }
atomic_write_json(ENVIRONMENT_PATH, TRAINING_ENVIRONMENT)
print(json.dumps(TRAINING_ENVIRONMENT, indent=2))


## 7. Load TRAIN and VALIDATION only; evaluate persistence first

In [ ]:
from ml.baseline import persistence_predictions
from ml.metrics import regression_metrics
from ml.data_loader import SplitArrays
from ml.artifacts import atomic_write_json

SEED = RANDOM_SEED
assert SEED == 42
LOAD_STARTED = time.perf_counter()
TRAIN_DATA = load_split_arrays(
    DATASET_DIR, "TRAIN", FEATURE_CONTRACT, expected_rows=EXPECTED_SPLIT_ROWS["TRAIN"]
)
VALIDATION_DATA = load_split_arrays(
    DATASET_DIR, "VALIDATION", FEATURE_CONTRACT, expected_rows=EXPECTED_SPLIT_ROWS["VALIDATION"]
)
DATA_LOAD_SECONDS = time.perf_counter() - LOAD_STARTED
BASELINE_VALIDATION_PREDICTION = persistence_predictions(
    VALIDATION_DATA.features, FEATURE_CONTRACT.model_features
)
BASELINE_VALIDATION_METRICS = regression_metrics(
    VALIDATION_DATA.target, BASELINE_VALIDATION_PREDICTION
)
atomic_write_json(RUN_DIR / "baseline_validation_metrics.json", BASELINE_VALIDATION_METRICS)
print({
    "train_rows": TRAIN_DATA.row_count,
    "validation_rows": VALIDATION_DATA.row_count,
    "load_seconds": DATA_LOAD_SECONDS,
    "persistence_validation": BASELINE_VALIDATION_METRICS,
})


## 8. Deterministic candidate search with Drive checkpoints

In [ ]:
from ml.xgboost_model import candidate_profiles, run_candidate_search

PLAN_PATH = RUN_DIR / "candidate_plan.json"
if PLAN_PATH.exists():
    CANDIDATE_CONFIGS = read_json(PLAN_PATH)
else:
    CANDIDATE_CONFIGS = candidate_profiles(ACCELERATOR["device"])
    atomic_write_json(PLAN_PATH, CANDIDATE_CONFIGS)

CANDIDATE_SEARCH_STARTED = time.perf_counter()
CANDIDATE_RESULTS = run_candidate_search(
    TRAIN_DATA,
    VALIDATION_DATA,
    FEATURE_CONTRACT.model_features,
    device=ACCELERATOR["device"],
    checkpoint_directory=str(RUN_DIR),
    nthread=os.cpu_count(),
    candidate_configs=CANDIDATE_CONFIGS,
)
CANDIDATE_SEARCH_SECONDS = time.perf_counter() - CANDIDATE_SEARCH_STARTED
candidate_table = pd.DataFrame(CANDIDATE_RESULTS)
display(candidate_table[["candidate_id", "best_iteration", "validation_metrics", "training_time_seconds", "device", "status"]])
print({
    "planned_candidate_count": len(CANDIDATE_CONFIGS),
    "successful_attempt_count": sum(row["status"] == "SUCCESS" for row in CANDIDATE_RESULTS),
    "candidate_search_seconds_this_session": CANDIDATE_SEARCH_SECONDS,
})


## 9. Select using VALIDATION only and freeze parameters

In [ ]:
from ml.xgboost_model import select_candidate
from ml.artifacts import atomic_write_json

SELECTED_CANDIDATE = select_candidate(CANDIDATE_RESULTS)
SELECTED_CANDIDATE["final_training_rows"] = EXPECTED_FINAL_TRAINING_ROWS
SELECTED_CANDIDATE["feature_list_sha256"] = FEATURE_CONTRACT.feature_list_sha256
SELECTED_CANDIDATE["training_code_commit"] = TRAINING_CODE_COMMIT
SELECTED_CANDIDATE["frozen_at_utc"] = datetime.now(timezone.utc).isoformat()
SELECTED_PATH = RUN_DIR / "selected_candidate.json"
if SELECTED_PATH.exists():
    persisted_selection = read_json(SELECTED_PATH)
    for field in ("candidate_id", "params", "best_iteration", "feature_list_sha256", "training_code_commit"):
        if persisted_selection.get(field) != SELECTED_CANDIDATE.get(field):
            raise RuntimeError(f"Resumed candidate selection changed frozen field {field}")
    SELECTED_CANDIDATE = persisted_selection
else:
    atomic_write_json(SELECTED_PATH, SELECTED_CANDIDATE)

SELECTED_VALIDATION_METRICS = SELECTED_CANDIDATE["validation_metrics"]
atomic_write_json(RUN_DIR / "selected_validation_metrics.json", SELECTED_VALIDATION_METRICS)
print(json.dumps(SELECTED_CANDIDATE, indent=2))
print("Selection is frozen. TEST has not been loaded.")


## 10. Final fit on all TRAIN + VALIDATION rows

In [ ]:
import gc

from ml.xgboost_model import fit_final_model
from ml.artifacts import atomic_write_json, sha256_file

MODEL_PATH = RUN_DIR / "weather_forecast_xgboost_v1.json"
FREEZE_PATH = RUN_DIR / "final_model_freeze.json"
if FREEZE_PATH.exists():
    MODEL_FREEZE = read_json(FREEZE_PATH)
    if MODEL_FREEZE["training_code_commit"] != TRAINING_CODE_COMMIT:
        raise RuntimeError("Frozen model was trained by a different code commit")
    if MODEL_FREEZE["feature_list_sha256"] != FEATURE_CONTRACT.feature_list_sha256:
        raise RuntimeError("Frozen model feature list differs from the current feature contract")
    if sha256_file(MODEL_PATH) != MODEL_FREEZE["model_sha256"]:
        raise RuntimeError("Frozen model file hash does not match its freeze record")
    FINAL_DEVICE = MODEL_FREEZE["device"]
    FINAL_FIT_SECONDS = MODEL_FREEZE["final_fit_seconds"]
    MODEL_FALLBACK_REASON = MODEL_FREEZE.get("device_fallback_reason")
    FINAL_BOOST_ROUNDS = int(MODEL_FREEZE["final_boost_rounds"])
    FINAL_MODEL = xgb.Booster()
    FINAL_MODEL.load_model(MODEL_PATH)
    RELOAD_VALIDATION = read_json(RUN_DIR / "model_reload_validation.json")
else:
    TRAINING_DATA = SplitArrays(
        np.concatenate([TRAIN_DATA.features, VALIDATION_DATA.features], axis=0),
        np.concatenate([TRAIN_DATA.target, VALIDATION_DATA.target], axis=0),
    )
    assert TRAINING_DATA.row_count == EXPECTED_FINAL_TRAINING_ROWS
    FINAL_BOOST_ROUNDS = int(SELECTED_CANDIDATE["best_iteration"]) + 1
    FINAL_FIT_STARTED = time.perf_counter()
    FINAL_MODEL, FINAL_DEVICE, MODEL_FALLBACK_REASON, FINAL_FIT_SECONDS = fit_final_model(
        TRAINING_DATA,
        FEATURE_CONTRACT.model_features,
        SELECTED_CANDIDATE,
        device=ACCELERATOR["device"],
        nthread=os.cpu_count(),
    )
    assert FINAL_MODEL.num_boosted_rounds() == FINAL_BOOST_ROUNDS
    FINAL_MODEL.save_model(MODEL_PATH)
    MODEL_SHA256 = sha256_file(MODEL_PATH)
    MODEL_BYTES = MODEL_PATH.stat().st_size

    sample_size = min(64, TRAINING_DATA.row_count)
    sample_matrix = xgb.DMatrix(
        TRAINING_DATA.features[:sample_size],
        feature_names=list(FEATURE_CONTRACT.model_features),
    )
    expected_sample_predictions = FINAL_MODEL.predict(
        sample_matrix, iteration_range=(0, FINAL_BOOST_ROUNDS)
    )
    reloaded_model = xgb.Booster()
    reloaded_model.load_model(MODEL_PATH)
    reloaded_sample_predictions = reloaded_model.predict(sample_matrix)
    RELOAD_RTOL = 1e-6
    RELOAD_ATOL = 1e-4
    max_absolute_difference = float(
        np.max(np.abs(expected_sample_predictions - reloaded_sample_predictions))
    )
    RELOAD_VALIDATION = {
        "status": "PASS" if np.allclose(
            expected_sample_predictions, reloaded_sample_predictions, rtol=RELOAD_RTOL, atol=RELOAD_ATOL
        ) else "FAIL",
        "sample_rows": sample_size,
        "max_absolute_prediction_difference": max_absolute_difference,
        "relative_tolerance": RELOAD_RTOL,
        "absolute_tolerance": RELOAD_ATOL,
        "model_sha256": MODEL_SHA256,
        "model_bytes": MODEL_BYTES,
        "fresh_booster_reload": True,
    }
    if RELOAD_VALIDATION["status"] != "PASS":
        raise RuntimeError("Fresh model reload validation failed")
    atomic_write_json(RUN_DIR / "model_reload_validation.json", RELOAD_VALIDATION)
    MODEL_FREEZE = {
        "status": "FROZEN",
        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
        "training_code_commit": TRAINING_CODE_COMMIT,
        "feature_list_sha256": FEATURE_CONTRACT.feature_list_sha256,
        "selected_candidate_id": SELECTED_CANDIDATE["candidate_id"],
        "selected_params": SELECTED_CANDIDATE["params"],
        "best_iteration": int(SELECTED_CANDIDATE["best_iteration"]),
        "final_boost_rounds": FINAL_BOOST_ROUNDS,
        "final_training_rows": TRAINING_DATA.row_count,
        "device": FINAL_DEVICE,
        "device_fallback_reason": MODEL_FALLBACK_REASON,
        "model_file": MODEL_PATH.name,
        "model_sha256": MODEL_SHA256,
        "model_bytes": MODEL_BYTES,
        "reload_validation": RELOAD_VALIDATION["status"],
        "final_fit_seconds": FINAL_FIT_SECONDS,
    }
    atomic_write_json(FREEZE_PATH, MODEL_FREEZE)
    del reloaded_model, sample_matrix, TRAINING_DATA

MODEL_SHA256 = MODEL_FREEZE["model_sha256"]
MODEL_BYTES = int(MODEL_FREEZE["model_bytes"])
assert RELOAD_VALIDATION["status"] == "PASS"
assert FINAL_BOOST_ROUNDS == int(SELECTED_CANDIDATE["best_iteration"]) + 1
for variable_name in ("TRAIN_DATA", "VALIDATION_DATA"):
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
print(json.dumps(MODEL_FREEZE, indent=2))
print("Model, feature order, parameters, rounds, code commit, and model hash are frozen before TEST.")


## 11. Official TEST 2025 evaluation — once after freeze

In [ ]:
from ml.evaluation import (
    evaluate_per_location,
    summarize_location_results,
    validate_test_coverage,
    write_test_predictions_parquet,
)
from ml.metrics import compare_metrics, regression_metrics

TEST_COMPLETE_PATH = RUN_DIR / "test_evaluation_complete.json"
TEST_STARTED_PATH = RUN_DIR / "test_evaluation_started.json"
if TEST_COMPLETE_PATH.exists():
    TEST_COMPLETION = read_json(TEST_COMPLETE_PATH)
    if TEST_COMPLETION["model_sha256"] != MODEL_SHA256:
        raise RuntimeError("Completed TEST metrics belong to a different frozen model")
    BASELINE_TEST_METRICS = read_json(RUN_DIR / "baseline_test_metrics.json")
    XGBOOST_TEST_METRICS = read_json(RUN_DIR / "xgboost_test_metrics.json")
    TEST_COMPARISON = read_json(RUN_DIR / "baseline_comparison.json")
    PER_LOCATION_ROWS = pd.read_csv(RUN_DIR / "per_location_test_metrics.csv").to_dict("records")
    LOCATION_SUMMARY = read_json(RUN_DIR / "location_summary.json")
    TEST_PREDICTION_SECONDS = TEST_COMPLETION["xgboost_prediction_seconds"]
    TEST_DATA_LOAD_SECONDS = TEST_COMPLETION["test_data_load_seconds"]
    print("TEST was already evaluated and its completion marker is valid; no TEST values were reloaded.")
else:
    retry_record = None
    test_attempt = 1
    if TEST_STARTED_PATH.exists():
        prior_attempt = read_json(TEST_STARTED_PATH)
        test_attempt = int(prior_attempt.get("attempt", 1)) + 1
        retry_record = {
            "prior_started_at_utc": prior_attempt["started_at_utc"],
            "reason": "Runtime stopped after the frozen model began TEST access and before a completion marker was saved. This is an artifact-recovery rerun; model selection and parameters remain frozen.",
        }
    atomic_write_json(
        TEST_STARTED_PATH,
        {
            "started_at_utc": datetime.now(timezone.utc).isoformat(),
            "attempt": test_attempt,
            "engineering_recovery": retry_record,
            "model_sha256": MODEL_SHA256,
            "training_code_commit": TRAINING_CODE_COMMIT,
        },
    )
    TEST_LOAD_STARTED = time.perf_counter()
    TEST_DATA = load_split_arrays(
        DATASET_DIR,
        "TEST",
        FEATURE_CONTRACT,
        include_metadata=True,
        expected_rows=EXPECTED_SPLIT_ROWS["TEST"],
    )
    TEST_DATA_LOAD_SECONDS = time.perf_counter() - TEST_LOAD_STARTED
    assert TEST_DATA.row_count == 551_880
    TEST_COVERAGE = validate_test_coverage(
        TEST_DATA.location_id, TEST_DATA.event_time, TEST_DATA.target_time
    )
    TEST_PERSISTENCE_PREDICTION = persistence_predictions(
        TEST_DATA.features, FEATURE_CONTRACT.model_features
    )
    test_matrix = xgb.DMatrix(
        TEST_DATA.features,
        feature_names=list(FEATURE_CONTRACT.model_features),
    )
    TEST_PREDICTION_STARTED = time.perf_counter()
    TEST_XGBOOST_PREDICTION = FINAL_MODEL.predict(
        test_matrix, iteration_range=(0, FINAL_BOOST_ROUNDS)
    )
    TEST_PREDICTION_SECONDS = time.perf_counter() - TEST_PREDICTION_STARTED

    BASELINE_TEST_METRICS = regression_metrics(TEST_DATA.target, TEST_PERSISTENCE_PREDICTION)
    XGBOOST_TEST_METRICS = regression_metrics(TEST_DATA.target, TEST_XGBOOST_PREDICTION)
    TEST_COMPARISON = compare_metrics(BASELINE_TEST_METRICS, XGBOOST_TEST_METRICS)
    PER_LOCATION_ROWS = evaluate_per_location(
        TEST_DATA.target,
        TEST_PERSISTENCE_PREDICTION,
        TEST_XGBOOST_PREDICTION,
        TEST_DATA.location_id,
    )
    LOCATION_SUMMARY = summarize_location_results(PER_LOCATION_ROWS)

    atomic_write_json(RUN_DIR / "baseline_test_metrics.json", BASELINE_TEST_METRICS)
    atomic_write_json(RUN_DIR / "xgboost_test_metrics.json", XGBOOST_TEST_METRICS)
    atomic_write_json(RUN_DIR / "baseline_comparison.json", TEST_COMPARISON)
    from ml.artifacts import write_csv
    write_csv(
        RUN_DIR / "per_location_test_metrics.csv",
        PER_LOCATION_ROWS,
        list(PER_LOCATION_ROWS[0].keys()),
    )
    atomic_write_json(RUN_DIR / "location_summary.json", LOCATION_SUMMARY)
    write_test_predictions_parquet(
        str(RUN_DIR / "test_predictions.parquet"),
        location_ids=TEST_DATA.location_id,
        event_times=TEST_DATA.event_time,
        target_times=TEST_DATA.target_time,
        actual=TEST_DATA.target,
        persistence_prediction=TEST_PERSISTENCE_PREDICTION,
        xgboost_prediction=TEST_XGBOOST_PREDICTION,
    )
    TEST_COMPLETION = {
        "status": "PASS",
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "official_test_evaluation_passes": test_attempt,
        "engineering_recovery": retry_record,
        "test_rows": TEST_DATA.row_count,
        "coverage": TEST_COVERAGE,
        "model_sha256": MODEL_SHA256,
        "training_code_commit": TRAINING_CODE_COMMIT,
        "test_data_load_seconds": TEST_DATA_LOAD_SECONDS,
        "xgboost_prediction_seconds": TEST_PREDICTION_SECONDS,
        "xgboost_prediction_rows_per_second": TEST_DATA.row_count / TEST_PREDICTION_SECONDS,
        "test_predictions_sha256": sha256_file(RUN_DIR / "test_predictions.parquet"),
        "test_predictions_bytes": (RUN_DIR / "test_predictions.parquet").stat().st_size,
    }
    atomic_write_json(TEST_COMPLETE_PATH, TEST_COMPLETION)
    del test_matrix

assert TEST_COMPLETION["status"] == "PASS"
assert BASELINE_TEST_METRICS["n"] == XGBOOST_TEST_METRICS["n"] == 551_880
print(json.dumps({
    "persistence_test": BASELINE_TEST_METRICS,
    "xgboost_test": XGBOOST_TEST_METRICS,
    "comparison": TEST_COMPARISON,
    "location_summary": {key: value for key, value in LOCATION_SUMMARY.items() if not key.startswith("worst_")},
}, indent=2))


## 12. Per-location diagnostics and feature importance

In [ ]:
from ml.xgboost_model import feature_importance_records
from ml.artifacts import write_csv

IMPORTANCE_ROWS = feature_importance_records(FINAL_MODEL, FEATURE_CONTRACT.model_features)
write_csv(
    RUN_DIR / "feature_importance.csv",
    IMPORTANCE_ROWS,
    ["feature", "gain", "weight"],
)
atomic_write_json(RUN_DIR / "worst_location_diagnostics.json", {
    "worst_10_xgboost_mae": LOCATION_SUMMARY["worst_10_xgboost_mae"],
    "worst_10_regression_vs_persistence": LOCATION_SUMMARY["worst_10_regression_vs_persistence"],
    "descriptive_only": True,
    "test_used_for_retuning": False,
})
display(pd.DataFrame(PER_LOCATION_ROWS).sort_values("mae_absolute_improvement").head(10))
display(pd.DataFrame(IMPORTANCE_ROWS).head(20))


## 13. Diagnostic plots (metrics remain authoritative)

In [ ]:
PLOT_DIR = RUN_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_PATH = RUN_DIR / "test_predictions.parquet"
if "TEST_DATA" in globals():
    plot_actual = TEST_DATA.target
    plot_prediction = TEST_XGBOOST_PREDICTION
    plot_error = TEST_XGBOOST_PREDICTION - TEST_DATA.target
else:
    plot_table = pd.read_parquet(PREDICTIONS_PATH)
    plot_actual = plot_table["actual_temperature_c"].to_numpy()
    plot_prediction = plot_table["xgboost_prediction_c"].to_numpy()
    plot_error = plot_table["xgboost_error_c"].to_numpy()

sample_indices = np.linspace(0, len(plot_actual) - 1, min(5_000, len(plot_actual)), dtype=int)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(plot_actual[sample_indices], label="Actual", linewidth=1)
ax.plot(plot_prediction[sample_indices], label="XGBoost", linewidth=1)
ax.set_title("TEST sample: actual vs XGBoost prediction")
ax.set_ylabel("Temperature (°C)")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "test_actual_vs_predicted.png", dpi=140)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(plot_error, bins=80, color="#2878B5", alpha=0.85)
ax.set_title("TEST XGBoost error distribution")
ax.set_xlabel("Prediction - actual (°C)")
ax.set_ylabel("Rows")
fig.tight_layout()
fig.savefig(PLOT_DIR / "test_error_distribution.png", dpi=140)
plt.show()
plt.close(fig)

location_frame = pd.DataFrame(PER_LOCATION_ROWS).sort_values("xgboost_mae")
fig, ax = plt.subplots(figsize=(13, 6))
positions = np.arange(len(location_frame))
ax.bar(positions - 0.2, location_frame["persistence_mae"], width=0.4, label="Persistence")
ax.bar(positions + 0.2, location_frame["xgboost_mae"], width=0.4, label="XGBoost")
ax.set_xticks(positions)
ax.set_xticklabels(location_frame["location_id"], rotation=90, fontsize=7)
ax.set_ylabel("MAE (°C)")
ax.set_title("TEST MAE by location")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "test_mae_by_location.png", dpi=140)
plt.show()
plt.close(fig)

importance_frame = pd.DataFrame(IMPORTANCE_ROWS).head(20).sort_values("gain")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(importance_frame["feature"], importance_frame["gain"], color="#34A853")
ax.set_xlabel("Gain importance")
ax.set_title("Top 20 feature importance by gain")
fig.tight_layout()
fig.savefig(PLOT_DIR / "feature_importance_gain_top20.png", dpi=140)
plt.show()
plt.close(fig)


## 14. Write the complete run manifest and checksums

In [ ]:
from ml.artifacts import atomic_write_json, sha256_file

MODEL_ARTIFACT_MANIFEST = {
    "model_file_name": MODEL_PATH.name,
    "drive_relative_path": str(MODEL_PATH.relative_to(DRIVE_MYDRIVE)).replace("\\", "/"),
    "sha256": MODEL_SHA256,
    "size_bytes": MODEL_BYTES,
    "run_id": RUN_ID,
    "model_id": "WEATHER_XGBOOST_GLOBAL_T1H_V1",
}
atomic_write_json(RUN_DIR / "model_artifact_manifest.json", MODEL_ARTIFACT_MANIFEST)

candidate_training_seconds = sum(
    float(row.get("training_time_seconds", 0.0) or 0.0) for row in CANDIDATE_RESULTS
)
TRAINING_MANIFEST = {
    "status": "COMPLETE",
    "model_id": "WEATHER_XGBOOST_GLOBAL_T1H_V1",
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_code_commit": TRAINING_CODE_COMMIT,
    "feature_set_id": FEATURE_CONTRACT.feature_set_id,
    "feature_manifest_path": FEATURE_CONTRACT.feature_manifest_path,
    "feature_manifest_sha256": FEATURE_CONTRACT.feature_manifest_sha256,
    "feature_spec_sha256": FEATURE_CONTRACT.feature_spec_sha256,
    "ml_schema_sha256": FEATURE_CONTRACT.ml_schema_sha256,
    "feature_list_sha256": FEATURE_CONTRACT.feature_list_sha256,
    "model_feature_count": len(FEATURE_CONTRACT.model_features),
    "target": TARGET_COLUMN,
    "train_rows": EXPECTED_SPLIT_ROWS["TRAIN"],
    "validation_rows": EXPECTED_SPLIT_ROWS["VALIDATION"],
    "final_training_rows": EXPECTED_FINAL_TRAINING_ROWS,
    "test_rows": EXPECTED_SPLIT_ROWS["TEST"],
    "seed": SEED,
    "candidate_count": len(CANDIDATE_CONFIGS),
    "candidate_success_count": sum(row["status"] == "SUCCESS" for row in CANDIDATE_RESULTS),
    "selected_candidate_id": SELECTED_CANDIDATE["candidate_id"],
    "selected_params": SELECTED_CANDIDATE["params"],
    "best_iteration": int(SELECTED_CANDIDATE["best_iteration"]),
    "final_boost_rounds": FINAL_BOOST_ROUNDS,
    "xgboost_version": xgb.__version__,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pyarrow_version": pyarrow.__version__,
    "sklearn_version": importlib.metadata.version("scikit-learn"),
    "device": FINAL_DEVICE,
    "device_fallback_reason": MODEL_FALLBACK_REASON,
    "gpu_model": ACCELERATOR["gpu_names"],
    "cpu_count": os.cpu_count(),
    "ram_bytes": TRAINING_ENVIRONMENT["ram_bytes"],
    "model_artifact_path": MODEL_ARTIFACT_MANIFEST["drive_relative_path"],
    "model_sha256": MODEL_SHA256,
    "model_bytes": MODEL_BYTES,
    "validation_metrics": SELECTED_VALIDATION_METRICS,
    "persistence_validation_metrics": BASELINE_VALIDATION_METRICS,
    "test_metrics": XGBOOST_TEST_METRICS,
    "persistence_test_metrics": BASELINE_TEST_METRICS,
    "test_comparison": TEST_COMPARISON,
    "location_summary": {key: value for key, value in LOCATION_SUMMARY.items() if not key.startswith("worst_")},
    "candidate_search_duration_seconds": candidate_training_seconds,
    "candidate_search_wall_seconds_this_session": CANDIDATE_SEARCH_SECONDS,
    "final_fit_duration_seconds": FINAL_FIT_SECONDS,
    "test_data_load_duration_seconds": TEST_DATA_LOAD_SECONDS,
    "test_prediction_duration_seconds": TEST_PREDICTION_SECONDS,
    "test_prediction_rows_per_second": TEST_COMPLETION["xgboost_prediction_rows_per_second"],
    "model_reload_validation": RELOAD_VALIDATION,
    "test_evaluation_completion": TEST_COMPLETION,
}
atomic_write_json(RUN_DIR / "training_manifest.json", TRAINING_MANIFEST)

COMMITTED_RESULT_FILES = [
    "model_feature_list.json",
    "training_code_commit.json",
    "dataset_verification.json",
    "training_environment.json",
    "candidate_plan.json",
    "candidate_results.csv",
    "candidate_results.json",
    "selected_candidate.json",
    "baseline_validation_metrics.json",
    "selected_validation_metrics.json",
    "baseline_test_metrics.json",
    "xgboost_test_metrics.json",
    "baseline_comparison.json",
    "per_location_test_metrics.csv",
    "location_summary.json",
    "worst_location_diagnostics.json",
    "feature_importance.csv",
    "model_artifact_manifest.json",
    "model_reload_validation.json",
    "final_model_freeze.json",
    "test_evaluation_started.json",
    "test_evaluation_complete.json",
    "training_manifest.json",
]
RESULT_CHECKSUMS = {
    "algorithm": "SHA-256",
    "scope": "Small committed modeling reports only; excludes this checksum file, model JSON, TEST prediction Parquet, and plots.",
    "files": {name: sha256_file(RUN_DIR / name) for name in COMMITTED_RESULT_FILES},
}
atomic_write_json(RUN_DIR / "checksums.json", RESULT_CHECKSUMS)
print(json.dumps({
    "run_id": RUN_ID,
    "training_code_commit": TRAINING_CODE_COMMIT,
    "model_sha256": MODEL_SHA256,
    "model_bytes": MODEL_BYTES,
    "candidate_training_seconds": candidate_training_seconds,
    "final_fit_seconds": FINAL_FIT_SECONDS,
    "test_prediction_seconds": TEST_PREDICTION_SECONDS,
    "result_files": len(COMMITTED_RESULT_FILES),
    "result_checksums": "PASS",
}, indent=2))


## 15. Verify result bundle and download small evidence only

In [ ]:
import zipfile
from google.colab import files

RESULT_BUNDLE = RUN_DIR / f"weather_forecast_xgboost_v1_results_{RUN_ID}.zip"
with zipfile.ZipFile(RESULT_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in [*COMMITTED_RESULT_FILES, "checksums.json"]:
        archive.write(RUN_DIR / name, arcname=name)

with zipfile.ZipFile(RESULT_BUNDLE, "r") as archive:
    invalid = archive.testzip()
    if invalid:
        raise RuntimeError(f"Corrupt result bundle member: {invalid}")
print({
    "result_bundle": str(RESULT_BUNDLE),
    "bundle_bytes": RESULT_BUNDLE.stat().st_size,
    "committed_result_count": len(COMMITTED_RESULT_FILES) + 1,
    "test_predictions_in_bundle": False,
    "model_binary_in_bundle": False,
})
files.download(str(RESULT_BUNDLE))


## 16. Final experiment checks

The notebook is complete only if every assertion below passes. The model and full TEST predictions remain in Drive; the downloadable ZIP contains small evidence only. A merge-ready research experiment does not mean the model is ready for streaming inference.


In [ ]:
assert TRAINING_MANIFEST["status"] == "COMPLETE"
assert TRAINING_MANIFEST["model_feature_count"] == 73
assert TRAINING_MANIFEST["final_training_rows"] == 2_760_849
assert TRAINING_MANIFEST["test_rows"] == 551_880
assert MODEL_FREEZE["status"] == "FROZEN"
assert RELOAD_VALIDATION["status"] == "PASS"
assert TEST_COMPLETION["status"] == "PASS"
assert len(PER_LOCATION_ROWS) == 63
assert TEST_COMPARISON["n"] == 551_880
print("PASS: frozen commit, feature order, temporal split, model reload, test evaluation, and all-location coverage.")
print("XGBoost beat persistence on TEST MAE:", XGBOOST_TEST_METRICS["mae"] < BASELINE_TEST_METRICS["mae"])
print("XGBoost beat persistence on TEST RMSE:", XGBOOST_TEST_METRICS["rmse"] < BASELINE_TEST_METRICS["rmse"])
print("Streaming inference is not part of this milestone.")
